<a href="https://colab.research.google.com/github/karkhutmaria/KarkhutM_NLP_hw/blob/main/%D0%9A%D0%B0%D1%80%D1%85%D1%83%D1%82_%D0%9C%2C_%D0%BB%D0%B0%D0%B1%D1%80%D0%B0%D0%B1_RNN%2C_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание: генерация текста с помощью RNN (собственный корпус)

## Задача

1. Собрать свой небольшой текстовый корпус (не менее 1000 предложений) с помощью библиотеки `requests` (парсинг новостного сайта, блога, форума и т.д.).
2. Обучить рекуррентную нейросеть (RNN/LSTM) на собранных данных для генерации текста (по образцу из приложенного ноутбука `Copy_of_rnn.ipynb`).
3. После обучения вывести на экран 2–3 сгенерированных предложения.
4. *Дополнительно (на 5 баллов, но не обязательно):* посчитать метрику перплексии (perplexity) на валидационной выборке.
5. *Для себя (не оценивается):* обучить модель с использованием GPU.

## Критерии оценки

- **3 балла** — корпус собран (≥1000 предложений), модель обучена, сгенерировано хотя бы 1 предложение.
- **4 балла** — всё из п.3 + код с комментариями, объясняющими ключевые этапы.
- **5 баллов** — всё из п.4 + дополнительно посчитана метрика перплексии.

## Важно

- Качество сгенерированного текста не оценивается.
- Выберите **свой уникальный сайт** для парсинга и укажите его в отчёте.
- Не используйте готовые датасеты из интернета.


## 1. Установка и импорт библиотек

In [1]:
!pip install beautifulsoup4 requests lxml -q

import requests
from bs4 import BeautifulSoup
import time
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow версия:", tf.__version__)
print("GPU доступна:", tf.config.list_physical_devices('GPU'))

TensorFlow версия: 2.20.0
GPU доступна: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Парсинг текстового корпуса

**Ваш уникальный сайт:** https://www.dailymail.com/travel/index.html

**Обоснование выбора:** чтобы генерировать заголовки для статей о путешествиях

In [10]:
def scrape_corpus(base_url, num_pages=5):

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    texts = []

    for page in range(1, num_pages + 1):
        url = f"{base_url}?page={page}"
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            for tag in soup.find_all(['h1', 'h2', 'h3', 'p']): #html-теги, необходимые для скрейпинга заголовков с сайта Daily Mail
                text = tag.get_text(strip=True)
                if text and len(text) > 10:
                    texts.append(text)

            print(f"Страница {page}: собрано {len(texts)} текстов")
            time.sleep(1)  # вежливый парсинг

        except Exception as e:
            print(f"Ошибка на странице {page}: {e}")

    return texts


base_url = "https://www.dailymail.com/travel/index.html"
corpus = scrape_corpus(base_url, num_pages=10)

print(f"\nВсего собрано текстов: {len(corpus)}")
print("Примеры:", corpus[:5])

Страница 1: собрано 215 текстов
Страница 2: собрано 430 текстов
Страница 3: собрано 645 текстов
Страница 4: собрано 860 текстов
Страница 5: собрано 1075 текстов
Страница 6: собрано 1290 текстов
Страница 7: собрано 1505 текстов
Страница 8: собрано 1720 текстов
Страница 9: собрано 1935 текстов
Страница 10: собрано 2150 текстов

Всего собрано текстов: 2150
Примеры: ['Travel News', "As the UK sees hottest May Bank Holiday on record, the country's best places for wild swimming walks", "Don't be snobbish about caravan park holidays! Metal roofs, grim shower blocks and bingo are out - and even celebs are flocking to enjoy hotel-level comfort", "Find your perfect all-inclusive Caribbean escape: From Jamaica and Antigua to Grenada and St Vincent, our guide to the island resort holiday that's right for you", 'Tranquillity, thrills, gourmet food and the great outdoors: Why Orlando is the best holiday you never dreamt of - til now']


## 3. Подготовка данных для RNN

Токенизация, создание последовательностей, паддинг.

In [11]:
# Создаём токенизатор
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

# Преобразуем тексты в последовательности чисел
sequences = tokenizer.texts_to_sequences(corpus)

# Создаём входные и выходные данные для causal language modeling
X, y = [], []
for seq in sequences:
    for i in range(1, len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

# Паддинг
X = pad_sequences(X)

# One-hot encoding для y
vocab_size = len(tokenizer.word_index) + 1
y = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

print(f"Размер входных данных X: {X.shape}")
print(f"Размер выходных данных y: {y.shape}")
print(f"Размер словаря: {vocab_size}")

Размер входных данных X: (44010, 41)
Размер выходных данных y: (44010, 1705)
Размер словаря: 1705


## 4. Создание и обучение модели RNN (LSTM)

In [35]:
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=100, input_length=X.shape[1]))
model.add(LSTM(150, return_sequences=False))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

history = model.fit(X, y, epochs=15, batch_size=32, validation_split=0.2) #делим выборку на тренировочную и валидационную

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
1101/1101 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.0691 - loss: 6.1184 - val_accuracy: 0.1132 - val_loss: 5.1737
Epoch 2/15
1101/1101 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.1781 - loss: 4.4161 - val_accuracy: 0.3195 - val_loss: 3.3996
Epoch 3/15
1101/1101 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.4722 - loss: 2.7476 - val_accuracy: 0.6969 - val_loss: 1.8871
Epoch 4/15
1101/1101 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.7704 - loss: 1.4534 - val_accuracy: 0.8716 - val_loss: 0.9399
Epoch 5/15
1101/1101 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.9030 - loss: 0.7276 - val_accuracy: 0.9411 - val_loss: 0.4777
Epoch 6/15
1101/1101 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.9516 - loss: 0.3811 - val_accuracy: 0.9648 - val_loss: 0.2685
Epoch 7/15
1101/1101 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.9675 - loss: 0.2222 - val_accuracy: 0.9718 - val_loss: 0.1663
Epoch 8/15
1101/1101 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.9712 - loss:

Как можно заметить по результатам эпох обучения, после 11 эпохи прогресс в обучении становится минимальным и иногда даже происходит откат назад.

## 5. Генерация текста

Функция генерации и вывод 2–3 предложений.

In [40]:
def generate_text(seed_text, next_words, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

# Пример генерации
seed = "UK"  # замените на любое начало
generated = generate_text(seed, next_words=8, max_sequence_len=X.shape[1])
print("Сгенерированный текст:", generated)

# Сгенерируйте ещё 1-2 примера с другими seed
seed2 = "Holidays"
generated2 = generate_text(seed2, next_words=7, max_sequence_len=X.shape[1])
print("Сгенерированный текст 2:", generated2)

seed3 = "Flight"  # замените на любое начало
generated3 = generate_text(seed3, next_words=3, max_sequence_len=X.shape[1])
print("Сгенерированный текст:", generated3)

Сгенерированный текст: UK beach closed to the public for 20 years
Сгенерированный текст 2: Holidays to turkey are half the price now
Сгенерированный текст: Flight prices are soaring


## 6. (Дополнительно, на 5 баллов) Расчёт перплексии

Перплексия = exp(loss). Чем ниже, тем лучше модель предсказывает последовательность.

In [42]:
# Оцениваем модель на валидационных данных
loss, accuracy = model.evaluate(X, y, verbose=0)
perplexity = np.exp(loss)

print(f"Потери (loss): {loss:.4f}")
print(f"Перплексия: {perplexity:.4f}")


Потери (loss): 0.0754
Перплексия: 1.0783


Метрика перплексии очень хорошая, она свидетельствует о том, что в большинстве случаев модель правильно предугадывает последующее слово. Вероятно, это в некоторой степени связано с тем, что газетные заголовки довольно однотипные, в них используются языковые штампы, что облегчает задачу модели по генерации заголовков

## 7. GPU (не оценивается)

Убедитесь, что обучение запускалось на GPU:


In [43]:
print("Устройства GPU:", tf.config.list_physical_devices('GPU'))

Устройства GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
